# MinerU `text_level` Refinement Agent

**Purpose**: Correct `text_level` tags in MinerU's `_content_list.json` output using Gemini 2.5 Flash VLM-based TOC extraction.

**Pipeline**:
1. **VLM TOC Extraction** — Send PDF to Gemini 2.5 Flash → extract section headings as structured JSON
2. **Human-in-the-Loop Review** — Display & confirm the LLM's TOC detection result before proceeding
3. **Fuzzy Match & Correct** — Use confirmed ground truth to fix `text_level` in `content_list.json`

**Dependencies**: `google-genai`, `rapidfuzz`, `PyMuPDF (fitz)`

In [5]:
import json
import os
import re
from pathlib import Path

from IPython.display import display, Markdown
from google import genai
from google.genai import types
from rapidfuzz import fuzz

# ============================================================
# CONFIGURATION — Edit these variables before running
# ============================================================
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
GEMINI_MODEL = 'gemini-2.5-flash'

DOC_CODE = "Advisory Guidelines on Key Concepts in the PDPA 17 May 2022"
PDF_PATH  = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\data privacy\{DOC_CODE}.pdf"
JSON_PATH = fr"C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\{DOC_CODE}\hybrid_auto\{DOC_CODE}_content_list copy.json"

OUTPUT_DIR = None  # Set to a directory path, or None to save alongside input JSON

# Human-in-the-loop temp file
HITL_JSON_PATH = None  # Auto-generated from JSON_PATH if None

print('Configuration loaded')
print(f'  Model: {GEMINI_MODEL}')
print(f'  PDF:   {PDF_PATH}')
print(f'  JSON:  {JSON_PATH}')

Configuration loaded
  Model: gemini-2.5-flash
  PDF:   C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\data privacy\Advisory Guidelines on Key Concepts in the PDPA 17 May 2022.pdf
  JSON:  C:\Users\User\Desktop\Compliance-GraphRAG\UnstructuredData_Transformation_Pipeline\out_md_mineru\Advisory Guidelines on Key Concepts in the PDPA 17 May 2022\hybrid_auto\Advisory Guidelines on Key Concepts in the PDPA 17 May 2022_content_list copy.json


In [6]:
# ============================================================
# Load content_list.json and show statistics
# ============================================================
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    content_list = json.load(f)

total_blocks = len(content_list)
text_level_blocks = [b for b in content_list if 'text_level' in b]
pages = set(b.get('page_idx', -1) for b in content_list)

print(f'Content List Statistics')
print(f'  Total blocks:          {total_blocks}')
print(f'  Blocks with text_level: {len(text_level_blocks)}')
print(f'  Page range:            {min(pages)} - {max(pages)}')
print(f'\n--- Current text_level blocks ---')
for i, b in enumerate(text_level_blocks):
    print(f'  [{i+1:3d}] page {b["page_idx"]:3d} | {b.get("text", "")[:80]}')

Content List Statistics
  Total blocks:          1460
  Blocks with text_level: 6
  Page range:            0 - 164

--- Current text_level blocks ---
  [  1] page   5 | PART IV: OFFENCES AFFECTING PERSONAL DATA AND ANONYMISED INFORMATION .156 
  [  2] page   6 | PART I: INTRODUCTION AND OVERVIEW 
  [  3] page   9 | PART II: IMPORTANT TERMS USED IN THE PDPA 
  [  4] page  33 | PART III: THE DATA PROTECTION PROVISIONS 
  [  5] page 155 | PART IV: OFFENCES AFFECTING PERSONAL DATA AND ANONYMISED INFORMATION 
  [  6] page 159 | PART V: OTHER RIGHTS, OBLIGATIONS AND USES 


## Stage 1: VLM TOC Extraction

Send the PDF document to **Gemini 2.5 Flash** to extract:
1. Which pages contain the Table of Contents (`toc_pages`, 0-indexed)
2. All **top-level** section headings listed in the TOC (`sections`)

The model receives the native PDF bytes and returns structured JSON.

In [ ]:
# ============================================================
# Stage 1: VLM TOC Extraction using Gemini 2.5 Flash
# ============================================================

# Initialize Gemini client
client = genai.Client(api_key=GEMINI_API_KEY)

# Read PDF bytes
with open(PDF_PATH, 'rb') as f:
    pdf_bytes = f.read()
print(f'PDF loaded: {len(pdf_bytes):,} bytes')

# System prompt for TOC extraction
SYSTEM_PROMPT = """
You are a Document Structure Analyst. Your task is to identify and extract the structural hierarchy from a document's Table of Contents (TOC).

RULES:
- Assign "level": 1 to the TOP-MOST heading tier in this document (e.g. CHAPTER, PART, TITLE, SECTION — whatever is the highest level).
- Assign "level": 2 to the SECOND tier headings that fall under level 1 (e.g. Article, Section, Clause).
- DO NOT include level 3 or below (sub-sections, sub-clauses, bullet points, indented descriptions).
- EXCLUDE page numbers, dot leaders (....), and extra whitespace.
- If the heading format in the TOC differs from the body content (e.g. TOC uses "Section 1 title" but body says "1. title"), ALWAYS  the exact wording and format as it appears in the body content, not the TOC.
- "heading": the structural label ONLY (e.g. "CHAPTER I", "Article 1", "Section 2", "1").
- "heading_name": the descriptive title ONLY (e.g. "General provisions"). If none exists, use null.

Output format:
{
  "sections": [
    {"level": 1, "heading": "CHAPTER I",  "heading_name": "General provisions"},
    {"level": 2, "heading": "Article 1",  "heading_name": "Subject-matter and objectives"},
    {"level": 1, "heading": "CHAPTER II", "heading_name": "Principles"},
    {"level": 2, "heading": "Article 5",  "heading_name": null}
  ]
}

Only output valid JSON.
"""

# Call Gemini 2.5 Flash
try:
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
            'Extract the Table of Contents structure from this PDF document. Return the result as JSON.'
        ],
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type='application/json',
            temperature=0,
        )
    )
except Exception as e:
    print(f'Error during Gemini API call: {e}')
    raise

# Parse result
raw_text = response.text
toc_result = json.loads(raw_text)

print(f'\nVLM TOC Extraction Complete')
print(f'  Sections found: {len(toc_result.get("sections", []))}')
print(f'\n--- Extracted Sections ---')
print(json.dumps(toc_result, indent=2))

PDF loaded: 3,349,022 bytes
Error during Gemini API call: Server disconnected without sending a response.


RemoteProtocolError: Server disconnected without sending a response.

In [ ]:
print(raw_text)

In [ ]:
# ============================================================
# Human-in-the-Loop: Review & Confirm TOC Result
# ============================================================

# Determine save path for the ground truth JSON
if HITL_JSON_PATH is None:
    json_dir = Path(JSON_PATH).parent
    HITL_JSON_PATH = str(json_dir / 'toc_ground_truth.json')

# Save LLM result to file for review / editing
with open(HITL_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(toc_result, f, indent=2, ensure_ascii=False)

print(f'Saved TOC result to: {HITL_JSON_PATH}')
print(f'\n{"="*70}')
print(f'LLM TOC Detection Result (review below)')
print(f'{"="*70}')
print(f'\nSections ({len(toc_result["sections"])} total):')
print(f'{"-"*70}')
for i, s in enumerate(toc_result['sections'], 1):
    sid = s.get('heading') or 'PART'
    print(f' {sid} {s["heading_name"]}')
print(f'{"-"*70}')

# ---- Confirmation gate ----
print(f'\nREVIEW the result above.')
print(f'If you need to edit, open the file at:\n  {HITL_JSON_PATH}')
confirm = input('\nType "yes" to confirm, or "edit" if you edited the file: ').strip().lower()

if confirm in ('edit', 'e'):
    # Reload the user-edited file
    with open(HITL_JSON_PATH, 'r', encoding='utf-8') as f:
        toc_result = json.load(f)
    print(f'\nReloaded edited file. Sections: {len(toc_result["sections"])}')
    # Show updated result
    for i, s in enumerate(toc_result['sections'], 1):
        sid = s.get('section_id') or ''
        print(f' {i:3d}. [{sid:>5}] {s["heading_name"]}')
elif confirm in ('yes', 'y'):
    print('\nConfirmed! Proceeding with correction...')
else:
    raise RuntimeError(f'Aborted. Got "{confirm}". Re-run this cell after review.')

# Store confirmed ground truth for Stage 2
# toc_pages = set(toc_result.get('toc_pages', []))
ground_truth_sections = toc_result['sections']
print(f'\nGround Truth: {len(ground_truth_sections)} sections')    

## Stage 2: Fuzzy Match & Correct `content_list.json`

Using the confirmed ground truth, apply corrections:

1. **Remove false positives** — Body blocks with `text_level: 1` that don't match any ground truth section
2. **Add false negatives** — Body blocks that match ground truth but currently lack `text_level`

In [ ]:
# ============================================================
# Stage 2: Fuzzy Matching & Correction Engine
# ============================================================

# Higher threshold since ground truth titles MUST exist in the body text
# MinerU errors are layout misclassification, not content errors
FUZZY_THRESHOLD = 95     # strict: titles should match very closely
TOKEN_SET_THRESHOLD = 92  # for token-set ratio pass (handles minor OCR gaps)
MIN_LEN_RATIO = 0.7       # block text length must be >= 70% of target length

def normalize(text):
    """Normalize text for comparison: lowercase, strip, collapse whitespace,
    remove trailing dots and page numbers."""
    # First defend
    # text = re.sub(r'[\.…]+\s*\d*\s*$', '', text)   # trailing dots + page nums
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text


def build_match_targets(sections):
    """Build match candidates from structured heading + heading_name fields.
    Only uses level 1 sections (no fallback to level 2).
    Each section produces up to 2 candidates:
      1. heading only   → e.g. "Section 2"
      2. heading + name → e.g. "Section 2 Customer acceptance policy"
    """
    heading_targets = []
    full_targets = []

    level1 = [s for s in sections if s.get('level') == 1]
    use_sections = level1 if level1 else [s for s in sections if s.get('level') == 2]

    for s in use_sections:
        heading      = s.get('heading', '') or ''
        heading_name = s.get('heading_name', '') or ''
        orig_title   = f"{heading} {heading_name}".strip()

        if heading:
            heading_targets.append((normalize(heading), orig_title))
        if heading and heading_name:
            full_targets.append((normalize(f'{heading} {heading_name}'), orig_title))

    return heading_targets, full_targets


def match_block(block_text, heading_targets, full_targets):
    """Check if a block's text matches any ground truth section.
    Pass 1: heading only     — e.g. block = "Section 2"
    Pass 2: heading + name   — e.g. block = "Section 2 Customer acceptance policy"
    Returns (matched, orig_title, score, method)
    """
    norm_text = normalize(block_text)
    if not norm_text:
        return False, None, 0, None

    # Pass 1: heading only
    best_score = 0
    best_title = None
    for norm_target, orig_title in heading_targets:
        score = fuzz.ratio(norm_text, norm_target)
        if score > best_score:
            best_score = score
            best_title = orig_title
        if score >= FUZZY_THRESHOLD:
            return True, orig_title, score, 'heading'

    # Pass 2: heading + heading_name (only if Pass 1 failed)
    for norm_target, orig_title in full_targets:
        score = fuzz.ratio(norm_text, norm_target)
        if score > best_score:
            best_score = score
            best_title = orig_title
        if score >= FUZZY_THRESHOLD:
            return True, orig_title, score, 'heading+name'

    
    
    return False, best_title, best_score, None



# Build match targets from confirmed ground truth
heading_targets, full_targets = build_match_targets(ground_truth_sections)

# Deep copy for correction
corrected = json.loads(json.dumps(content_list))
corrections = {'added': [], 'removed': [], 'kept': [], 'toc_removed': []}

for i, block in enumerate(corrected):
    page = block.get('page_idx', -1)
    page = int(page)+1
    has_level = 'text_level' in block
    block_text = block.get('text', '')
    block_type = block.get('type', '')

    # Skip non-text block types (image, table, page_number, header, etc.)
    if block_type not in ('text',):
        if has_level:
            del block['text_level']
            corrections['removed'].append({
                'index': i, 'page': page, 'text': block_text[:80],
                'reason': f'non-text block type: {block_type}'
            })
        continue
    
    # Body page blocks: match against ground truth
    matched, match_title, score, method = match_block(block_text, heading_targets, full_targets)

    if has_level and matched:
        # Correct — keep text_level
        corrections['kept'].append({
            'index': i, 'page': page, 'text': block_text[:80],
            'matched': match_title, 'score': score, 'method': method
        })
    elif has_level and not matched:
        # False positive — MinerU incorrectly tagged this as heading
        del block['text_level']
        corrections['removed'].append({
            'index': i, 'page': page, 'text': block_text[:80],
            'reason': f'no ground truth match (best score: {score})'
        })
    elif not has_level and matched:
        # False negative — MinerU missed this heading
        block['text_level'] = 1
        corrections['added'].append({
            'index': i, 'page': page, 'text': block_text[:80],
            'matched': match_title, 'score': score, 'method': method
        })

# ---- Summary ----
print('Correction Complete')
print('=' * 70)
print(f'  Kept (correct):        {len(corrections["kept"])}')
print(f'  Removed (false pos):   {len(corrections["removed"])}')
print(f'  Removed (TOC pages):   {len(corrections["toc_removed"])}')
print(f'  Added (false neg):     {len(corrections["added"])}')
print('=' * 70)

if corrections['removed']:
    print('\n--- Removed (False Positives) ---')
    for c in corrections['removed']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]} | reason: {c["reason"]}')

if corrections['toc_removed']:
    print('\n--- Removed (TOC Page Entries) ---')
    for c in corrections['toc_removed']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]}')

if corrections['added']:
    print('\n--- Added (False Negatives) ---')
    for c in corrections['added']:
        print(f'  page {c["page"]:3d} | {c["text"][:60]} | matched: {c["matched"]} ({c["score"]} via {c["method"]})')

In [ ]:
# ============================================================
# Save corrected JSON and correction log
# ============================================================

# Overwrite original file directly (avoid long path issue)
original_path = Path(JSON_PATH)
output_path = original_path.parent / (original_path.stem + '_corrected' + original_path.suffix)
log_path = Path(JSON_PATH).parent / 'correction_log.json'

# Save corrected content_list (overwrite original)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(corrected, f, indent=4, ensure_ascii=False)

# Save correction log for auditability
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump({
        'source': str(JSON_PATH),
        'ground_truth': str(HITL_JSON_PATH),
        'stats': {
            'total_blocks': len(corrected),
            'kept': len(corrections['kept']),
            'removed_false_pos': len(corrections['removed']),
            'removed_toc': len(corrections['toc_removed']),
            'added_false_neg': len(corrections['added']),
        },
        'corrections': corrections,
    }, f, indent=2, ensure_ascii=False)

# Before vs After comparison
original_levels = sum(1 for b in content_list if 'text_level' in b)
corrected_levels = sum(1 for b in corrected if 'text_level' in b)

print(f'Saved corrected JSON:  {output_path}')
print(f'Saved correction log:  {log_path}')
print(f'\nBefore -> After:')
print(f'  text_level blocks: {original_levels} -> {corrected_levels}')
print(f'  Net change: {corrected_levels - original_levels:+d}')

In [ ]:
# ============================================================
# Validation Spot-Check
# ============================================================

# Show all text_level blocks in the corrected output
final_levels = [b for b in corrected if 'text_level' in b]

print(f'Final text_level Blocks ({len(final_levels)} total)')
print('=' * 70)
for i, b in enumerate(final_levels, 1):
    page = b.get('page_idx', '?')
    text = b.get('text', '')[:70]
    print(f'  {i:3d}. page {page:3d} | {text}')
print('=' * 70)

# Cross-check: which ground truth sections were NOT matched in the document?
matched_titles = set()
for c in corrections['kept'] + corrections['added']:
    matched_titles.add(c.get('matched', ''))

unmatched = [s for s in ground_truth_sections if s['heading_name'] not in matched_titles]

if unmatched:
    print(f'\nWARNING: Ground truth sections NOT found in content_list.json ({len(unmatched)}):')
    for s in unmatched:
        print(f'{s["heading_name"]}')
else:
    print(f'\nAll ground truth sections matched in content_list.json')